# Find the own-rule ablation layer
One shared implementation for S1, voice, clause order and lexical rules. Extract and intervene after the SAME supplied space in `Final answer: `. Fit on 50 discovery pairs; choose the layer on 30 separate selection pairs; test only the selected layer on the preserved evaluation pairs.

The mean direction is the baseline. For a broader own-rule subspace set METHOD='pooled' and RANK=2 or 4 before running. Base-prediction preservation gates layer selection; five random directions/subspaces of matching rank are evaluated. Free generation is a separate test; do not equate fixed-cue decoding with restored semantic reasoning. These previously inspected datasets make this an exploratory study.


In [ ]:
import torch
assert torch.cuda.is_available(), "Select a GPU runtime"
%pip install -q "transformers==4.49.0" "peft==0.14.0" "datasets<4" accelerate pyyaml tqdm matplotlib plotly
from google.colab import files
from pathlib import Path
import os,sys,zipfile,json,hashlib
ROOT=Path('/content/stegano_experiments'); ROOT.mkdir(exist_ok=True)
print('Upload stegano_experiments_bundle.zip')
uploaded=files.upload()
with zipfile.ZipFile(next(n for n in uploaded if n.endswith('.zip'))) as z:
    for name in z.namelist(): assert (ROOT/name).resolve().is_relative_to(ROOT.resolve())
    z.extractall(ROOT)
os.chdir(ROOT);sys.path.insert(0,str(ROOT))
for name,digest in json.loads(Path('bundle_manifest.json').read_text()).items():
    assert hashlib.sha256(Path(name).read_bytes()).hexdigest()==digest,name
from experiments.data import config
from experiments.colab import ensure_adapters,download
cfg=config()


In [ ]:
RULE='clause'  # set lexical after training and upload its adapter when prompted
LAYERS=list(range(24))  # zero-based Qwen2.5-0.5B blocks; use [18] for a single layer
METHOD='mean'
RANK=1
RUN_FREE_GENERATION=True
OUT=Path(f'{RULE}_residual_scan_v2')
ensure_adapters(cfg,[RULE])


In [ ]:
from experiments.residual import scan
selected=scan(cfg,RULE,LAYERS,OUT,rank=RANK,method=METHOD)
download(OUT)  # Preserve fixed results before optional free generation.


In [ ]:
from experiments.residual import free_generation
if RUN_FREE_GENERATION:
    free_generation(cfg,selected,OUT/'free')
    download(OUT)
